# BERTopic Benchmark
Run BERTopic topic modelling with full evaluation metrics: **NMI**, **Purity**, **Precision/Recall/F1**, **Topic Diversity**, and **Coherence (Cᵥ)**. Includes exports for CSV, Top-Words Bar Chart, UMAP 3D Scatter Plot, Ground Truth 3D Scatter, and BERTopic Interactive HTMLs.

## 1. Installation
Run this cell once to install all required dependencies. Skip if already installed.

In [13]:
import sys

!{sys.executable} -m pip install \
    notebook \
    ipywidgets \
    numpy \
    matplotlib \
    seaborn \
    tqdm \
    scikit-learn \
    spacy \
    gensim \
    bertopic \
    umap-learn \
    pandas \
    plotly \
    sentence-transformers

# Download spaCy English model
!{sys.executable} -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 115.1 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## 2. Imports

In [1]:
import csv
import json
import time
import logging
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import umap as umap_learn
import spacy
import warnings

from umap import UMAP
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import normalized_mutual_info_score
from gensim.corpora.dictionary import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from google.colab import drive


warnings.filterwarnings("ignore", category=DeprecationWarning)

drive.mount('/content/drive')

logging.basicConfig(level=logging.WARNING)
print("Imports OK.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Imports OK.


## 3. Configuration
Set all options here before running the rest of the notebook.

In [14]:
# --- Input ---
INPUT_FILE             = '/content/drive/MyDrive/ColabData/dataset_comsci_math.json'   # Path to JSON dataset file
K                      = None             # Reduce to K topics after fitting. None = auto

# --- BERTopic mode ---
USE_APPROX_DIST        = True           # True = c-TF-IDF soft clustering; False = HDBSCAN probabilities
USE_LEMMATIZED_INPUT   = False            # True = lemmatize text before embedding
REDUCE_OUTLIERS        = False             # True = reassign outlier docs (-1) to nearest topic via embeddings
OUTLIER_THRESHOLD      = 0.5             # Similarity threshold for outlier reassignment (0.0–1.0, higher = stricter)

# --- Label thresholds ---
THRESHOLD              = 0.3             # OpenAlex concept score threshold
ABS_THRESHOLD          = 0.25            # Absolute probability threshold for multi-label prediction
REL_THRESHOLD          = 0.1             # Relative threshold multiplier vs. dominant topic
TARGET_LEVEL           = 0               # Concept level (0, 1, or 2)

# --- Ground truth scatter options ---
GROUP_MULTI            = False            # True = collapse multi-label papers into one colour

# --- Exports (set to None to skip) ---
EXPORT_CSV             = 'bertopic_results.csv'          # Per-paper CSV
EXPORT_BARCHART        = 'bertopic_bar.png'              # Top-words bar chart
EXPORT_SCATTER_3D      = 'bertopic_scatter_3d.html'      # Predicted-label UMAP 3D
EXPORT_TRUE_SCATTER_3D = 'bertopic_true_scatter_3d.html' # Ground-truth UMAP 3D
EXPORT_HTML_PREFIX     = 'bertopic_html'                 # Prefix for BERTopic interactive HTMLs

## 4. BERTopic Service
Self-contained class wrapping BERTopic training, outlier reduction, and all export methods.

In [15]:
class BERTopicService:
    def __init__(self, n_topics=None, use_approx_dist=False, use_lemmatized_input=False,
                 reduce_outliers=True, outlier_threshold=0.5):
        self.n_topics             = n_topics
        self.use_approx_dist      = use_approx_dist
        self.use_lemmatized_input = use_lemmatized_input
        self.reduce_outliers      = reduce_outliers
        self.outlier_threshold    = outlier_threshold
        self.topic_model          = None
        self.topics               = None
        self.probs                = None

        self.nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])
        self.nlp.max_length = 10_000_000

        academic_stopwords = [
            'finding', 'findings', 'illustrate', 'significant', 'provide', 'provides',
            'potential', 'associated', 'effective', 'aspect', 'aspects', 'challenge', 'challenges',
            'paper', 'study', 'research', 'result', 'results', 'method', 'methodology',
            'proposed', 'propose', 'approach', 'based', 'using', 'used', 'use', 'to', 'we', 'source',
            'analysis', 'model', 'system', 'data', 'application', 'new', 'development',
            'performance', 'conclusion', 'abstract', 'introduction', 'work', 'time',
            'shown', 'show', 'demonstrate', 'experiment', 'experimental',
            'university', 'department', 'author', 'et', 'al', 'figure', 'table',
            'high', 'low', 'large', 'small', 'different', 'various', 'property', 'properties',
            'increase', 'effect', 'activity', 'structure', 'compound', 'condition', 'quality',
            'entry', 'contain', 'parameter', 'observe', 'report', 'present', 'evaluate'
        ]
        self.custom_stopwords = list(set(list(ENGLISH_STOP_WORDS) + academic_stopwords))

    def spacy_tokenizer(self, text):
        if not text:
            return []
        doc = self.nlp(text)
        return [
            token.lemma_.lower() for token in doc
            if not token.is_stop
            and not token.is_punct
            and not token.like_num
            and len(token) > 2
        ]

    def fit_transform(self, documents):
        print(f"Training BERTopic (Target Topics: {self.n_topics if self.n_topics else 'Auto'})...")
        mode_str = 'approximate_distribution (c-TF-IDF)' if self.use_approx_dist else 'calculate_probabilities (HDBSCAN)'
        print(f"Mode  : {mode_str}")

        if self.use_lemmatized_input:
            print("Input : Lemmatized Text")
            train_docs = [" ".join(self.spacy_tokenizer(doc)) for doc in documents]
        else:
            print("Input : Raw Text")
            train_docs = documents

        # BERTopic feeds per-cluster subsets into the vectorizer, not the full corpus.
        # Using min_df > 1 or max_df < 1.0 on small clusters causes ValueError.
        # Safe fix: min_df=1, max_df=1.0 — stopwords handle noise instead.
        self.vectorizer_model = CountVectorizer(
            stop_words=self.custom_stopwords,
            ngram_range=(1, 2),
            min_df=0.7,
            max_df=5
        )

        umap_model = UMAP(
            n_neighbors=min(15, len(train_docs) - 1),  # n_neighbors must be < n_docs
            n_components=5,
            min_dist=0.0, metric='cosine',
            random_state=42, low_memory=True
        )
        self.topic_model = BERTopic(
            umap_model=umap_model,
            embedding_model="allenai/specter2_base",
            vectorizer_model=self.vectorizer_model,
            calculate_probabilities=not self.use_approx_dist,
            nr_topics=self.n_topics if self.n_topics else None,
            verbose=True,
            top_n_words=15
        )
        self.topics, self.probs = self.topic_model.fit_transform(train_docs)

        if self.reduce_outliers:
            outlier_count = sum(1 for t in self.topics if t == -1)
            print(f"Reducing outliers (embeddings strategy, threshold={self.outlier_threshold})..."
                  f" [{outlier_count} outlier docs before reduction]")
            new_topics = self.topic_model.reduce_outliers(
                train_docs, self.topics, strategy="embeddings",
                threshold=self.outlier_threshold
            )
            self.topic_model.update_topics(
                train_docs, topics=new_topics, vectorizer_model=self.vectorizer_model
            )
            self.topics = new_topics
            remaining = sum(1 for t in self.topics if t == -1)
            print(f"Outlier reduction complete. [{remaining} docs still unassigned]")
        else:
            print("Outlier reduction skipped (REDUCE_OUTLIERS=False). "
                  f"[{sum(1 for t in self.topics if t == -1)} docs remain as outliers (-1)]")

        if self.use_approx_dist:
            print("Calculating approximate topic distributions (c-TF-IDF)...")
            topic_distr, _ = self.topic_model.approximate_distribution(train_docs, min_similarity=0.01)
            row_sums = topic_distr.sum(axis=1)
            row_sums[row_sums == 0] = 1
            self.probs = topic_distr / row_sums[:, np.newaxis]

        return self.topics, self.probs

    def get_top_words_list(self, n_top_words=15):
        n_valid = len(self.probs[0]) if self.probs is not None and len(self.probs) > 0 else 0
        topics_words = []
        for t_id in range(n_valid):
            if t_id in self.topic_model.get_topics():
                topics_words.append([w for w, _ in self.topic_model.get_topic(t_id)[:n_top_words]])
            else:
                topics_words.append([])
        return topics_words

    def calculate_topic_diversity(self, n_top_words=15):
        topics_words = self.get_top_words_list(n_top_words)
        unique_words = set(w for t in topics_words if t for w in t)
        total_words  = sum(len(t) for t in topics_words if t)
        return len(unique_words) / total_words if total_words else 0

    def calculate_coherence_score(self, documents):
        print("Calculating BERTopic Coherence (Cᵥ)...")
        tokenized_docs = [self.spacy_tokenizer(doc) for doc in documents]
        topics_words   = [t for t in self.get_top_words_list(15) if t]
        dictionary     = Dictionary(tokenized_docs)
        try:
            cm = CoherenceModel(
                topics=topics_words, texts=tokenized_docs,
                dictionary=dictionary, coherence='c_v'
            )
            return cm.get_coherence()
        except Exception as e:
            print(f"Coherence calc error: {e}")
            return 0.0

    def export_top_words_barchart(self, output_path, n_top_words=15):
        print("Generating Top Words Bar Chart...")
        valid_topics = [t for t in self.topic_model.get_topics().keys() if t != -1]
        n_topics = len(valid_topics)
        if n_topics == 0:
            print("No valid topics to plot."); return

        cols = 3
        rows = int(np.ceil(n_topics / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
        axes = np.array([axes]).flatten() if not isinstance(axes, np.ndarray) else axes.flatten()

        for idx, t_idx in enumerate(valid_topics):
            topic_data   = self.topic_model.get_topic(t_idx)
            top_features = [w for w, _ in topic_data[:n_top_words]]
            weights      = [s for _, s in topic_data[:n_top_words]]
            ax = axes[idx]
            ax.barh(top_features, weights, align='center', color='mediumpurple')
            ax.invert_yaxis()
            ax.set_title(f'Topic {t_idx}', fontdict={'fontsize': 14, 'fontweight': 'bold'})
            ax.tick_params(axis='both', which='major', labelsize=10)

        for i in range(n_topics, len(axes)):
            fig.delaxes(axes[i])

        plt.tight_layout()
        plt.savefig(output_path, dpi=300)
        plt.close()
        print(f"Bar chart saved → {output_path}")

    def _build_cluster_legend_map(self, cluster_ids, n_top_words_for_legend=5):
        legend_map = {}
        for t_idx in set(cluster_ids):
            if t_idx == -1:
                legend_map[t_idx] = "Topic -1: Outliers / Noise"
            else:
                topic_data = self.topic_model.get_topic(t_idx)
                if topic_data:
                    kw = ", ".join([w for w, _ in topic_data[:n_top_words_for_legend]])
                    legend_map[t_idx] = f"Topic {t_idx}: {kw}"
                else:
                    legend_map[t_idx] = f"Topic {t_idx}: Unknown"
        return legend_map

    def export_document_scatter_3d(self, output_path, cluster_ids, n_top_words_for_legend=5):
        if self.probs is None: return
        print("Running UMAP for 3D visualization (this may take a moment)...")
        reducer    = umap_learn.UMAP(n_components=3, random_state=42, metric='cosine')
        embedding  = reducer.fit_transform(self.probs)
        legend_map = self._build_cluster_legend_map(cluster_ids, n_top_words_for_legend)

        df = pd.DataFrame({
            'UMAP_1': embedding[:, 0], 'UMAP_2': embedding[:, 1], 'UMAP_3': embedding[:, 2],
            'Cluster Focus': [legend_map[cid] for cid in cluster_ids]
        })
        fig = px.scatter_3d(
            df, x='UMAP_1', y='UMAP_2', z='UMAP_3', color='Cluster Focus',
            title='BERTopic Document Clusters (UMAP 3D Projection)',
            opacity=0.8, color_discrete_sequence=px.colors.qualitative.Alphabet
        )
        fig.update_traces(marker=dict(size=4, line=dict(width=0.5, color='white')))
        fig.update_layout(legend=dict(itemsizing='constant'))
        fig.write_html(output_path)
        print(f"UMAP 3D scatter saved → {output_path}")

    def export_ground_truth_scatter_3d(self, output_path, true_labels_list):
        if self.probs is None: return
        print("Running UMAP for Ground Truth 3D visualization...")
        reducer   = umap_learn.UMAP(n_components=3, random_state=42, metric='cosine')
        embedding = reducer.fit_transform(self.probs)

        df = pd.DataFrame({
            'UMAP_1': embedding[:, 0], 'UMAP_2': embedding[:, 1], 'UMAP_3': embedding[:, 2],
            'Ground Truth': true_labels_list
        })
        fig = px.scatter_3d(
            df, x='UMAP_1', y='UMAP_2', z='UMAP_3', color='Ground Truth',
            title='Document Clusters (Colored by Ground Truth Labels)',
            opacity=0.8, color_discrete_sequence=px.colors.qualitative.Plotly
        )
        fig.update_traces(marker=dict(size=4, line=dict(width=0.5, color='white')))
        fig.update_layout(legend=dict(itemsizing='constant'))
        fig.write_html(output_path)
        print(f"Ground truth 3D scatter saved → {output_path}")

    def export_bertopic_html(self, output_prefix):
        if self.topic_model is None: return
        print("Exporting BERTopic interactive HTMLs...")
        try:
            valid_topics = [t for t in self.topic_model.get_topics().keys() if t != -1]
            n_valid      = len(valid_topics)
            if n_valid >= 3:
                self.topic_model.visualize_topics().write_html(f"{output_prefix}_distance.html")
            if n_valid >= 1:
                self.topic_model.visualize_barchart(top_n_topics=min(10, n_valid)).write_html(f"{output_prefix}_barchart.html")
            if n_valid >= 2:
                self.topic_model.visualize_heatmap().write_html(f"{output_prefix}_heatmap.html")
                self.topic_model.visualize_hierarchy().write_html(f"{output_prefix}_hierarchy.html")
            print(f"BERTopic HTMLs saved → {output_prefix}_*.html")
        except Exception as e:
            print(f"Failed to export BERTopic HTMLs: {e}")

print("BERTopicService defined.")

BERTopicService defined.


## 5. Load Data

In [16]:
target_key_hard  = f'true_label_l{TARGET_LEVEL}'
target_key_multi = f'multi_labels_l{TARGET_LEVEL}'

documents        = []
papers_data      = []
y_true_dominant  = []

print(f"Loading data from: {INPUT_FILE}")
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)

for item in data:
    text = item.get('text', '')
    if not text:
        text = f"{item.get('title', '')} {item.get('abstract', '')}"

    true_labels_set = set()
    top_label = item.get(target_key_hard)

    if target_key_multi in item and isinstance(item[target_key_multi], list):
        true_labels_set = set(item[target_key_multi])
    elif 'openalex_concepts' in item:
        valid_concepts = [
            c for c in item['openalex_concepts']
            if c.get('level') == TARGET_LEVEL and c.get('score', 0) >= THRESHOLD
        ]
        true_labels_set.update([c['name'] for c in valid_concepts])
        if valid_concepts and not top_label:
            valid_concepts.sort(key=lambda x: x.get('score', 0), reverse=True)
            top_label = valid_concepts[0]['name']
    else:
        if top_label:
            true_labels_set.add(top_label)

    if true_labels_set and top_label and text.strip():
        documents.append(text)
        y_true_dominant.append(top_label)
        papers_data.append({
            "id":          str(item.get('id', 'N/A')),
            "title":       str(item.get('title', 'Unknown Title')).replace('\n', ' ').replace('\r', ''),
            "true_labels": list(true_labels_set),
            "top_label":   str(top_label)
        })

print(f"Loaded {len(documents)} documents.")

Loading data from: /content/drive/MyDrive/ColabData/dataset_comsci_math.json
Loaded 264 documents.


## 6. Train BERTopic
⚠️ Downloads `allenai/specter2_base` (~500 MB) on first run.

In [17]:
start_time = time.time()

bertopic_service = BERTopicService(
    n_topics=K,
    use_approx_dist=USE_APPROX_DIST,
    use_lemmatized_input=USE_LEMMATIZED_INPUT,
    reduce_outliers=REDUCE_OUTLIERS,
    outlier_threshold=OUTLIER_THRESHOLD
)
topics, probs = bertopic_service.fit_transform(documents)

execution_time = time.time() - start_time

if probs is None:
    raise RuntimeError("fit_transform returned None for probs. Check BERTopic configuration.")

n_valid_topics     = len(probs[0])
topics_words_list  = bertopic_service.get_top_words_list(n_top_words=10)
topic_keywords_map = {i: ", ".join(words) for i, words in enumerate(topics_words_list)}
topic_keywords_map[-1] = "Outlier"

print(f"Training complete in {execution_time:.2f}s — {n_valid_topics} valid topics found.")

2026-03-23 06:22:32,018 - BERTopic - Embedding - Transforming documents to embeddings.


Training BERTopic (Target Topics: Auto)...
Mode  : approximate_distribution (c-TF-IDF)
Input : Raw Text


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/specter2_base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

2026-03-23 06:22:37,317 - BERTopic - Embedding - Completed ✓
2026-03-23 06:22:37,317 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-23 06:22:38,127 - BERTopic - Dimensionality - Completed ✓
2026-03-23 06:22:38,129 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-23 06:22:38,145 - BERTopic - Cluster - Completed ✓
2026-03-23 06:22:38,149 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-23 06:22:38,208 - BERTopic - Representation - Completed ✓


Outlier reduction skipped (REDUCE_OUTLIERS=False). [6 docs remain as outliers (-1)]
Calculating approximate topic distributions (c-TF-IDF)...


100%|██████████| 1/1 [00:00<00:00,  4.51it/s]

Training complete in 6.99s — 3 valid topics found.


## 7. Compute Metrics

In [18]:
# --- NMI & Purity ---
nmi = normalized_mutual_info_score(y_true_dominant, topics)

cluster_to_label_map = {}
for cid in set(topics):
    indices = [i for i, x in enumerate(topics) if x == cid]
    if indices:
        labels_in_cluster = [y_true_dominant[i] for i in indices]
        cluster_to_label_map[cid] = Counter(labels_in_cluster).most_common(1)[0][0]
    else:
        cluster_to_label_map[cid] = "Unknown"

y_pred_mapped = [cluster_to_label_map.get(cid, "Unknown") for cid in topics]
purity = sum(1 for yt, yp in zip(y_true_dominant, y_pred_mapped) if yt == yp) / len(y_true_dominant)

# --- Per-paper Precision / Recall / F1 ---
f1_scores, precision_scores, recall_scores = [], [], []

for i, paper_item in enumerate(papers_data):
    true_labels_set = set(paper_item["true_labels"])
    doc_probs = probs[i]
    max_prob  = max(doc_probs) if len(doc_probs) > 0 else 0
    pred_labels = set()

    for t_id, prob in enumerate(doc_probs):
        if prob > ABS_THRESHOLD and prob >= (max_prob * REL_THRESHOLD):
            mapped = cluster_to_label_map.get(t_id, "Unknown")
            if mapped != "Unknown":
                pred_labels.add(mapped)

    hard_cluster_id = int(topics[i])
    if not pred_labels:
        pred_labels.add(cluster_to_label_map.get(hard_cluster_id, "Unknown"))

    intersection = len(true_labels_set & pred_labels)
    p  = intersection / len(pred_labels)    if pred_labels    else 0
    r  = intersection / len(true_labels_set) if true_labels_set else 0
    f1 = 2 * (p * r) / (p + r)             if (p + r) > 0    else 0

    precision_scores.append(p)
    recall_scores.append(r)
    f1_scores.append(f1)

    paper_item.update({
        "predicted_top_topic_id":   str(hard_cluster_id),
        "predicted_top_label":      str(cluster_to_label_map.get(hard_cluster_id, "Unknown")),
        "predicted_multi_labels":   list(pred_labels),
        "predicted_topic_keywords": topic_keywords_map.get(hard_cluster_id, ""),
        "precision":                p,
        "recall":                   r,
        "f1_score":                 f1,
        "topic_distribution":       [float(prob) for prob in doc_probs]
    })

avg_f1    = np.mean(f1_scores)
avg_p     = np.mean(precision_scores)
avg_r     = np.mean(recall_scores)
diversity = bertopic_service.calculate_topic_diversity()
coherence = bertopic_service.calculate_coherence_score(documents)

print("Metrics computed.")

Calculating BERTopic Coherence (Cᵥ)...
Metrics computed.


## 8. Results Summary

In [19]:
mode_str  = "(Approx Dist)" if USE_APPROX_DIST else "(HDBSCAN Prob)"
lemma_str = "[LEMMATIZED INPUT]" if USE_LEMMATIZED_INPUT else "[RAW TEXT]"

print("=" * 60)
print(f"BERTopic BENCHMARK RESULTS {mode_str} {lemma_str} - Level {TARGET_LEVEL}")
print("=" * 60)
print(f"{'NMI Score (Single)':<30} | {nmi:.4f}")
print(f"{'Purity (Single)':<30} | {purity:.4f}")
print("-" * 60)
print(f"{'Avg Precision (Multi)':<30} | {avg_p:.4f}")
print(f"{'Avg Recall (Multi)':<30} | {avg_r:.4f}")
print(f"{'Avg F1 Score (Multi)':<30} | {avg_f1:.4f}")
print("-" * 60)
print(f"{'Topic Diversity':<30} | {diversity:.4f}")
print(f"{'Topic Coherence (Cv)':<30} | {coherence:.4f}")
print("-" * 60)
print(f"Execution Time: {execution_time:.2f} seconds")
print("=" * 60)

print(f"\nBERTopic TOPIC MAPPING (First {min(10, n_valid_topics)} Valid Topics)")
print("-" * 60)
for cid in range(min(10, n_valid_topics)):
    label = cluster_to_label_map.get(cid, "Unknown")
    words = ", ".join(topics_words_list[cid]) if cid < len(topics_words_list) else ""
    print(f"Topic {cid:<2} -> {label:<25} | Keywords: {words}")

BERTopic BENCHMARK RESULTS (Approx Dist) [RAW TEXT] - Level 0
NMI Score (Single)             | 0.5045
Purity (Single)                | 0.9205
------------------------------------------------------------
Avg Precision (Multi)          | 0.8068
Avg Recall (Multi)             | 0.9160
Avg F1 Score (Multi)           | 0.8299
------------------------------------------------------------
Topic Diversity                | 0.9556
Topic Coherence (Cv)           | 0.4876
------------------------------------------------------------
Execution Time: 6.99 seconds

BERTopic TOPIC MAPPING (First 3 Valid Topics)
------------------------------------------------------------
Topic 0  -> Computer science          | Keywords: models, algorithm, techniques, test, thailand, framework, problem, technique, methods, datasets
Topic 1  -> Mathematics               | Keywords: confidence, intervals, interval, bootstrap, confidence intervals, distribution, confidence interval, simulation, coverage, methods
Topic 2  ->

## 9. Export CSV

In [ ]:
if EXPORT_CSV:
    with open(EXPORT_CSV, 'w', encoding='utf-8-sig', newline='') as f:
        writer = csv.writer(f, quoting=csv.QUOTE_MINIMAL)
        header = [
            'Paper ID', 'Title', 'True Top Label', 'True Multi-Labels',
            'Predicted Top Topic ID', 'Predicted Top Label', 'Predicted Multi-Labels',
            'Predicted Topic Keywords', 'Precision', 'Recall', 'F1-Score'
        ]
        for t in range(n_valid_topics):
            header.append(f"Topic_{t}_Prob")
        writer.writerow(header)

        for p in papers_data:
            row = [
                p.get('id', ''),
                p.get('title', ''),
                p.get('top_label', ''),
                ", ".join([str(x) for x in p.get('true_labels', [])]),
                p.get('predicted_top_topic_id', ''),
                p.get('predicted_top_label', ''),
                ", ".join([str(x) for x in p.get('predicted_multi_labels', [])]),
                p.get('predicted_topic_keywords', ''),
                f"{p.get('precision', 0):.4f}",
                f"{p.get('recall', 0):.4f}",
                f"{p.get('f1_score', 0):.4f}"
            ]
            row.extend([f"{prob:.4f}" for prob in p.get('topic_distribution', [])])
            writer.writerow(row)

    print(f"CSV exported → {EXPORT_CSV}")
else:
    print("CSV export skipped (EXPORT_CSV is None).")

## 10. Export Top-Words Bar Chart

In [ ]:
if EXPORT_BARCHART:
    bertopic_service.export_top_words_barchart(EXPORT_BARCHART)
else:
    print("Bar chart export skipped (EXPORT_BARCHART is None).")

## 11. Export UMAP 3D Scatter Plot (Predicted Labels)

In [ ]:
if EXPORT_SCATTER_3D:
    bertopic_service.export_document_scatter_3d(EXPORT_SCATTER_3D, topics)
else:
    print("Predicted-label 3D scatter skipped (EXPORT_SCATTER_3D is None).")

## 12. Export UMAP 3D Scatter Plot (Ground Truth Labels)

In [ ]:
if EXPORT_TRUE_SCATTER_3D:
    true_labels_for_plot = []
    for p in papers_data:
        labels = p.get('true_labels', [])
        if len(labels) > 1:
            if GROUP_MULTI:
                true_labels_for_plot.append("Multi-label (Interdisciplinary)")
            else:
                true_labels_for_plot.append(" + ".join(sorted([str(l) for l in labels])))
        elif len(labels) == 1:
            true_labels_for_plot.append(str(labels[0]))
        else:
            true_labels_for_plot.append("Unknown")

    bertopic_service.export_ground_truth_scatter_3d(EXPORT_TRUE_SCATTER_3D, true_labels_for_plot)
else:
    print("Ground truth 3D scatter skipped (EXPORT_TRUE_SCATTER_3D is None).")

## 13. Export BERTopic Interactive HTMLs
Generates up to 4 files: `_distance.html`, `_barchart.html`, `_heatmap.html`, `_hierarchy.html`.

In [ ]:
if EXPORT_HTML_PREFIX:
    bertopic_service.export_bertopic_html(EXPORT_HTML_PREFIX)
else:
    print("BERTopic HTML export skipped (EXPORT_HTML_PREFIX is None).")